# Stochastic Simulation — Detailed Summary

## Chapter 1 — Introduction

### 1.1 The Sampling Problem
- **Goal**: draw X^(i) ~ p*, i=1,…,N accurately
- Often only have unnormalised p̄*(x) with p*(x) = p̄*(x)/Z, where Z = ∫p̄*(x)dx is intractable
- Notation: (φ, p*) = E_{p*}[φ(X)] = ∫φ(x)p*(x)dx

### Applications
1. **Simulation of complex systems** (forward simulation)
2. **Bayesian statistical inference** (conditioning on data to find hidden variables)
3. **Generative modelling** (sampling from unknown distribution given dataset)

### Motivating Example: Estimating π
- Circle inscribed in square [−1,1]²: P(B) = π/4 where B = {(x,y): x²+y² ≤ 1}
- Monte Carlo: P(B) = E_P[1_B(X)] ≈ N_c/N → π/4
- Converges as N → ∞

## Chapter 1 — Bayesian Inference Primer

### 1.3.1 Bayes' Rule
**p(x|y) = p(y|x)p(x) / p(y)**

| Component | Name | Role |
|-----------|------|------|
| p(x) | Prior | Encodes prior knowledge about latent variable |
| p(y\|x) | Likelihood | Model of observation process |
| p(x\|y) | Posterior | Updated belief after data |
| p(y) | Marginal likelihood / Evidence | Normalising constant; model comparison |

**Remark 1.1**: Maps to sampling framework — target p*(x) := p(x|y), unnormalised p̄*(x) := p(y|x)p(x), normalising constant Z := p(y).

### Example 1.1 — Gaussian Conjugacy (1D)
Prior: p(x) = N(μ₀, σ₀²), Likelihood: p(y|x) = N(y; x, σ²)

**Posterior**: p(x|y) = N(μ_p, σ_p²) where:
- μ_p = (σ²μ₀ + σ₀²y) / (σ² + σ₀²)
- σ_p² = σ²σ₀² / (σ² + σ₀²)

Derivation: expand exp{−(y−x)²/(2σ²) − (x−μ₀)²/(2σ₀²)}, complete the square in x.

### Example 1.2 — Multivariate Gaussian Conjugacy
Prior: p(x) = N(μ₀, Σ₀), Likelihood: p(y|x) = N(Ax, Σ)

**Posterior**: x|y ~ N(μ_p, Σ_p) where:
- Σ_p = (A^TΣ⁻¹A + Σ₀⁻¹)⁻¹
- μ_p = Σ_p(A^TΣ⁻¹y + Σ₀⁻¹μ₀)

**Woodbury alternative**:
- Σ_p = Σ₀ − Σ₀A^T(AΣ₀A^T + Σ)⁻¹AΣ₀
- μ_p = μ₀ + Σ₀A^T(AΣ₀A^T + Σ)⁻¹(y − Aμ₀)

### Example 1.3 — Poisson-Gamma Conjugacy
Prior: Gamma(α, β), Likelihood: Pois(y; x)

**Posterior**: Gamma(α + y, β + 1)

### Proposition 1.1 — Conditional Bayes Rule
p(x|y,z) = p(y|x,z)p(x|z) / p(y|z)

### 1.3.2 Conditional Independence
**Definition 1.1**: X ⊥ Y | Z iff p(x,y|z) = p(x|z)p(y|z)

Bayes update with conditionally i.i.d. observations:
- p(x|y_{1:n}) ∝ p(x) ∏ᵢ p(yᵢ|x)

### 1.3.3 Marginal Likelihood
p(y) = ∫p(y|x)p(x)dx

**Example 1.4** (Gaussian): p(y) = N(y; μ₀, σ₀² + σ²)

**Example 1.5** (Model comparison): compare p₀(y) vs p₁(y) — model with higher marginal likelihood is preferred for given y.

## Chapter 2 — Generating Uniform Random Variates

### Pseudo-Random Numbers (Definition 2.1)
Deterministic sequence whose statistical properties match true random sequence from desired distribution. Advantages: repeatable, fast, cheap.

### Linear Congruential Generator (LCG)
- x_{n+1} ≡ ax_n + b (mod m)
- u_n = x_n/m ∈ [0,1)
- x₀ = seed, m = modulus, a = multiplier, b = shift
- b = 0: multiplicative; b ≠ 0: mixed generator
- Period T ≤ m; goal: T = m (full period)
- m ~ 2³² for single precision
- **Shortcomings**: periodic, fails in high dimensions (lattice structure in (u_k, u_{k+1}) plots)
- Modern standard: **Mersenne-Twister** (not an LCG)

## Chapter 2 — Inverse Transform Method

### Theorem 2.1 (Probability Integral Transform)
If X has CDF F_X, then F_X⁻¹(U) with U ~ Unif(0,1) has the same distribution as X.

**Proof**: P(F_X⁻¹(U) ≤ x) = P(U ≤ F_X(x)) = F_X(x). ∎

**Algorithm 1**: For i=1,…,n: generate U_i ~ Unif(0,1), set X_i = F_X⁻¹(U_i).

### Worked Examples

| Distribution | CDF F(x) | Inverse F⁻¹(u) | Sampler |
|-------------|----------|-----------------|--------|
| Discrete on {s₁,…,s_K} with weights w_k | F(s_k) = Σ_{j=1}^k w_j | Find min k: F(s_k) ≥ U | Staircase inversion |
| Exp(λ) | 1 − e^{−λx} | −(1/λ)ln(1−U) | U~Unif, X = −ln(1−U)/λ |
| Cauchy | ½ + π⁻¹arctan(x) | tan(π(U − ½)) | U~Unif, X = tan(π(U−½)) |
| Poisson(λ) | e^{−λ}Σ_{i=0}^k λⁱ/i! | Smallest k: F(k) ≥ U | Sequential search |
| Weibull(k) | 1 − e^{−x^k} | (−ln(1−U))^{1/k} | U~Unif, X = (−ln(1−U))^{1/k} |

**Limitation**: requires closed-form F⁻¹ (e.g. no closed form for Gaussian CDF).

## Chapter 2 — Transformation Method

### General Principle
Design g such that X = g(U) has target distribution. Need the transformation of random variables formula:

**1D**: p_Y(y) = p_X(g⁻¹(y)) |dg⁻¹(y)/dy|

**nD**: p_Y(y) = p_X(g⁻¹(y)) |det J_{g⁻¹}(y)|, where J_{g⁻¹} is the Jacobian of the inverse.

### Example 2.5 — Deriving the Box-Müller Transform
Start with X₁ ~ Exp(1/2), X₂ ~ Unif(−π,π). Transform:
- Y₁ = √X₁ cos X₂, Y₂ = √X₁ sin X₂
- Inverse: x₁ = y₁²+y₂², x₂ = arctan(y₂/y₁)
- Jacobian: |det J_{g⁻¹}| = 2
- Result: p_{Y₁,Y₂} = N(y₁;0,1)N(y₂;0,1) ✓

### Box-Müller Algorithm (Section 2.2.3)
Let U₁, U₂ ~ Unif(0,1) independent. Then:
- Z₁ = √(−2 ln U₁) cos(2πU₂)
- Z₂ = √(−2 ln U₁) sin(2πU₂)
are independent N(0,1). Standard method for Gaussian sampling in practice.

### Example 2.6 — Uniform on Disk
r ~ Unif(0,1), θ ~ Unif(−π,π):
- X₁ = √r cos θ, X₂ = √r sin θ → Uniform on unit disk (density = 1/π)
- **Caution**: X₁ = r cos θ (without √r) gives non-uniform distribution p = 1/(2π√(x₁²+x₂²))

### Example 2.7 — N(μ, σ²) from N(0,1)
g(x) = σx + μ. Verify via 1D transformation formula: p_Y(y) = N(y; μ, σ²). ✓

## Chapter 2 — Composition Methods

### 2.3.1 Sampling Joint Distributions
**Proposition 2.1 (Chain Rule for Sampling)**: To sample (X₁,X₂) ~ p(x₁,x₂):
1. X₁ ~ p_{X₁}(·)
2. X₂ | X₁=x₁ ~ p_{X₂|X₁}(·|x₁)

For n variables: sample sequentially X₁, then X₂|X₁, then X₃|X₁,X₂, etc.

**Example 2.8 (Linear Model)**: p(x)=Unif(−10,10), p(y|x)=N(ax+b, σ²) → generate (x,y) pairs with linear relationship + noise.

### 2.3.2 Sampling Marginals
**Proposition 2.2 (Marginalisation by Projection)**: Sample from p(x₁,x₂), keep only X₁ → samples from marginal p(x₁).

**Proof**: P(Z ∈ A) = ∫_{ℝ^{d₁}} ∫_A p(x₁,x₂) dx₁ dx₂ = ∫_A p_{X₂}(x₂) dx₂. ∎

### Discrete Mixture Sampling
For p*(x) = Σ_k w_k q_k(x):
1. k ~ Categorical(w₁,…,w_K) (using inversion)
2. X ~ q_k(x)

### 2.4 Multivariate Gaussian Sampling
To sample Y ~ N(μ, Σ) where Σ ∈ ℝ^{d×d}:
1. Cholesky: Σ = LL^T
2. v = [v₁,…,v_d]^T with each v_k ~ N(0,1) independently
3. Y = μ + Lv

Generalises 1D: Y = μ + σX where X ~ N(0,1).

## Chapter 2 — Rejection Sampling

### Theorem 2.2 (Fundamental Theorem of Simulation)
Sampling X ~ p*(x) is equivalent to sampling (X,Y) uniformly on the region A = {(x,y): 0 ≤ y ≤ p̄*(x)} and keeping the x-marginal.

**Proof**: Joint q(x,y) = (1/Z)1_A(x,y). Marginal: q(x) = ∫₀^{p̄*(x)} (1/Z)dy = p̄*(x)/Z = p*(x). ∎

Works for unnormalised densities p̄* too.

### Rejection Sampling Algorithm
Choose proposal q(x) and constant M such that p̄*(x) ≤ Mq(x) for all x.

**Algorithm 5** (normalised) / **Algorithm 6** (unnormalised):
1. Sample X' ~ q(x)
2. Sample U ~ Unif(0,1)
3. If U ≤ p̄*(X')/(Mq(X')): **accept** X'; else: **reject**

### Proposition 2.3 (Acceptance Rate)
- Normalised target: **â = 1/M** (where M ≥ 1)
- Unnormalised target: **â = Z/M** (where Z = ∫p̄*dx)

**Proof**: â = E[a(X')] = ∫(p̄*(x)/(Mq(x)))q(x)dx = Z/M. For normalised: Z=1. ∎

### Optimal M and Proposal Design
- **Optimal M**: M* = sup_x p*(x)/q(x) (smallest covering constant)
  - Find x* = argmax p*/q (take log, differentiate, set to zero)
  - Then M* = p*(x*)/q(x*)
- **Optimal proposal**: parameterise q_θ, then θ* = argmin_θ log M_θ

### Key Examples

| Example | Target | Proposal | Key Result |
|---------|--------|----------|------------|
| 2.10 | Beta(2,2) | Box [0,1]×[0,1.5] | â = 1/1.5 = 0.67 |
| 2.11 | Beta(2,2) | N(0.5, 0.25) | M=1.3, â = 0.77 (better) |
| 2.12 | Truncated N(0,1) on [−a,a] | N(0,1) | M=1, â = Z = ∫_{−a}^a N(y;0,1)dy |
| 2.13 | Gaussian posterior | N(0,1) | Rejection from prior without computing posterior |
| 2.14 | exp(−x²/2) | Cauchy | M = 2e^{−1/2}, x*=±1 |
| 2.15 | Gamma(α,1) | Exp(λ) | Optimal λ*=1/α, M*=e^{−(α−1)}/Γ(α) |
| 2.16 | Poisson-Gamma posterior | Exp(λ) | Optimal λ*=2/(α+y) |

## Chapter 2 — Curse of Dimensionality

### Why Rejection Fails in High Dimensions
For unit ball inside unit cube:
- â_d = Vol(ball) / Vol(cube) = π^{d/2} / Γ(d/2+1)
- â₂ ≈ 0.78, â₃ ≈ 0.52, and decays **super-exponentially** in d

### General Result
If p*(x) = ∏p₀(xᵢ) and q(x) = ∏q₀(xᵢ) with K = sup p₀/q₀, then M = K^d → acceptance rate â = 1/K^d → 0 as d → ∞.

### Practical Issues
- Product ∏p(yᵢ|X') underflows for moderate n → use **log-space**: log a = log p̄*(X') − log M − log q(X'), accept when log U ≤ log a
- Global bound M may not exist for complex posteriors

### Mitigation
- If product structure exists: sample coordinate-by-coordinate → cost O(dK) instead of O(K^d)
- Motivates advanced methods: MCMC, SMC (covered in later chapters)

## Chapter 3 — Monte Carlo Integration

### 3.1 MC Estimator
- **Goal**: estimate (φ, p*) = E_{p*}[φ(X)] = ∫φ(x)p*(x)dx
- **MC estimator**: φ̂_N = (1/N) Σ_{i=1}^N φ(X^(i)), where X^(i) ~ p* i.i.d.
- **Prop 3.1 (Unbiasedness)**: E[φ̂_N] = E_{p*}[φ(X)] for any N
- **Prop 3.2 (Variance)**: Var(φ̂_N) = σ²/N where σ² = Var_{p*}(φ(X))
  - MSE = Var(φ̂_N) = σ²/N (since unbiased, MSE = variance)
  - Convergence: O(1/√N) regardless of dimension — but sampling from p* may be hard in high-d
- **CLT**: √N(φ̂_N − μ) →_d N(0, σ²) as N→∞
  - 95% CI: φ̂_N ± 1.96σ/√N

### 3.2 Error Metrics
| Metric | Formula | Notes |
|--------|---------|-------|
| Bias | E[θ̂] − θ | Zero for MC estimator |
| Variance | E[(θ̂ − E[θ̂])²] | σ²/N for MC |
| MSE | Bias² + Variance | = Variance when unbiased |
| RMSE | √MSE | Convergence rate O(1/√N) |

Connection to Dirac delta: φ̂_N = (φ, p^N) where p^N = (1/N)Σδ_{X^(i)} is the empirical measure.

## Chapter 3 — Importance Sampling

### 3.3.1 Basic IS
- **Problem**: can't sample from p* but can evaluate p̄*(x) and sample from proposal q(x)
- **IS estimator**: Ê_N = (1/N) Σ W^(i)φ(X^(i)), where W^(i) = p̄*(X^(i))/q(X^(i)), X^(i)~q
- **Prop 3.3**: E[Ê_N] = Z · E_{p*}[φ(X)], where Z = ∫p̄*dx — so it estimates Z·(φ,p*), not (φ,p*) directly
- If p* normalised (Z=1): IS estimator with W(x)=p*(x)/q(x) is unbiased for E_{p*}[φ]

### 3.3.2 Self-Normalised IS (SNIS)
- When Z unknown, use normalised weights: w̄^(i) = W^(i) / Σ_j W^(j)
- **SNIS estimator**: φ̂_SNIS = Σ w̄^(i) φ(X^(i))
- **Properties**: biased but consistent (converges to true value as N→∞)
- Particle approximation: p^N(dx) = Σ w̄^(i) δ_{X^(i)}(dx)

### 3.3.3 Optimal IS Proposal
- **Prop 3.5**: The variance-minimising proposal is q*(x) ∝ |φ(x)|p*(x)
- In practice q* is usually unavailable (requires knowing p*)
- For estimating Z: optimal q*(x) ∝ p*(x) (trivial — back to original problem)
- Practical approach: use parametric family q_θ, tune θ to minimise variance

### Variance of IS
- Var_q[Wφ] can be much larger or smaller than σ² depending on q
- IS helps when q places more mass where |φ|p* is large
- IS hurts when q has lighter tails than p* (W(x) explodes in tails)

## Chapter 3 — Bayesian IS & Implementation

### 3.4 Bayesian Inference with IS
- Target: p(x|y) ∝ p(y|x)p(x); unnormalised p̄* = p(y|x)p(x)
- IS weights: W^(i) = p(y|X^(i))p(X^(i))/q(X^(i))
- Special case — **prior as proposal**: q(x) = p(x) → W^(i) = p(y|X^(i)) (just the likelihood)
- SNIS with prior proposal: w̄^(i) ∝ p(y|X^(i)); intuition: upweight samples that explain the data well
- Marginal likelihood estimate: p̂(y) = (1/N)Σ W^(i) = (1/N)Σ p(y|X^(i)) when q = prior

### 3.5 Implementation Details

**Log-weight trick** (numerical stability):
1. Compute log W^(i) = log p̄*(X^(i)) − log q(X^(i))
2. Find max: L = max_i log W^(i)
3. Normalise: w̄^(i) = exp(log W^(i) − L) / Σ_j exp(log W^(j) − L)

**SIR (Sampling-Importance-Resampling)**:
1. Draw X^(i) ~ q, compute normalised weights w̄^(i)
2. Resample: draw N indices from Categorical(w̄^(1),…,w̄^(N))
3. Output: unweighted (equally weighted) approximate samples from p*

**Effective Sample Size (Def 3.1)**:
- ESS = (Σ W^(i))² / Σ(W^(i))² (equivalently 1/Σ(w̄^(i))²)
- ESS ∈ [1, N]: N = perfect (uniform weights), 1 = useless (one dominant weight)
- Rule of thumb: resample when ESS < N/2

**Mixture IS** (for robustness):
- Use q(x) = Σ_k α_k q_k(x), a mixture of proposals
- Avoids light-tail problem of single proposal
- Deterministic mixture weights α_k can be optimised

## Chapter 4 — Markov Chain Fundamentals

### 4.1 Discrete Markov Chains
- **Def 4.1 (Markov chain)**: P(X_t|X_{0:t-1}) = P(X_t|X_{t-1}); memoryless
- **Def 4.2 (Transition matrix)**: M_{ij} = P(X_t=j|X_{t-1}=i); rows sum to 1
- **n-step transition**: M^n gives n-step probabilities; p_n = p₀ M^n
- **Irreducibility**: every state reachable from every other state
- **Recurrence**: state i is recurrent if chain returns to i with probability 1; transient otherwise
- **Invariant distribution**: p* such that p* = p*M (left eigenvector with eigenvalue 1)
- **Detailed balance**: p*(i)M_{ij} = p*(j)M_{ji} for all i,j ⟹ p* is invariant
- **Convergence** (ergodic theorem): positive recurrence + aperiodicity ⟹ p_n → p* as n→∞

### 4.2 Continuous State Space
- **Transition kernel**: K(x'|x); replaces transition matrix
- **K-invariance** (Def 4.3): p*(x') = ∫K(x'|x)p*(x)dx
- **Detailed balance** (Def 4.4): K(x'|x)p*(x) = K(x|x')p*(x') for all x,x'
- **Prop 4.1**: Detailed balance ⟹ K-invariance (integrate both sides over x)
- Convergence: φ-irreducibility + aperiodicity + Harris recurrence → convergence to p*
- **AR(1) example**: X_t = φX_{t-1} + σε_t; if |φ|<1, invariant is N(0, σ²/(1−φ²))

## Chapter 4 — Metropolis-Hastings

### Algorithm 9 (MH)
Given target p̄*(x) (unnormalised OK) and proposal q(x'|x):
1. Initialise x₀
2. For t=1,…,T: propose x' ~ q(·|x_{t-1}), compute α = min(1, p̄*(x')q(x_{t-1}|x')/(p̄*(x_{t-1})q(x'|x_{t-1})))
3. Accept x_t = x' with prob α, else x_t = x_{t-1}

### Prop 4.2 — MH Satisfies Detailed Balance
- MH kernel: K(x'|x) = q(x'|x)α(x,x') + δ(x'−x)r(x), where r(x) = 1 − ∫q(x'|x)α(x,x')dx'
- Proof: K(x'|x)p*(x) = q(x'|x)α(x,x')p*(x) = min(q(x'|x)p*(x), q(x|x')p*(x')) = K(x|x')p*(x') ✓
- Works with unnormalised p̄* since normalising constants cancel in ratio

### Proposal Variants

| Variant | Proposal q(x'|x) | Acceptance ratio simplification |
|---------|------------------|--------------------------------|
| Independent MH | q(x'|x) = q(x') | α = min(1, p̄*(x')q(x)/(p̄*(x)q(x'))) |
| Random walk MH | q(x'|x) = q(x|x') (symmetric) | α = min(1, p̄*(x')/p̄*(x)) |
| MALA | N(x + γ∇log p*, 2γI) | Full ratio needed (q is not symmetric) |

### §4.3.4 Bayesian Inference with MH
- Target: p(x|y) ∝ p(y|x)p(x); set p̄* = p(y|x)p(x) in MH
- **Algorithm 10**: same as Alg 9 with p̄* = likelihood × prior
- Examples: source localisation (2D position), unknown mean + variance

## Chapter 4 — Gibbs Sampling

### Algorithm 11 (Deterministic Scan Gibbs)
Target p*(x₁,…,x_d). At each iteration, cycle:
- For m = 1,…,d: sample x_m^(t) ~ p_m(x_m | x_1^(t),…,x_{m-1}^(t), x_{m+1}^(t-1),…,x_d^(t-1))

No accept/reject step — every proposal is accepted.

### Prop 4.3 — Gibbs Stationarity
- Each kernel K_m(x'|x) = p_m*(x'_m|x_{-m})δ(x'_{-m}=x_{-m}) satisfies detailed balance w.r.t. p*
- Proof: K_m(x'|x)p*(x) = p_m*(x'_m|x_{-m})δ(x'_{-m}=x_{-m})p*(x) = p*(x'_m,x_{-m})δ(x'_{-m}=x_{-m}) — symmetric in x↔x'
- Composite K = K_1∘…∘K_d: each K_m leaves p* invariant, so does K (by induction)

### Algorithm 12 (Random Scan Gibbs)
- At each step, pick m uniformly from {1,…,d}, update only x_m from its full conditional
- Also preserves p* as invariant distribution

### Key Examples
- **Image denoising**: binary pixel grid, Gibbs with local conditionals based on neighbours
- **Beta-Binomial**: X|P ~ Bin(n,P), P ~ Beta(α,β) → full conditionals: P|X ~ Beta(α+X, β+n−X), X|P ~ Bin(n,P)

### §4.4.1 Metropolis-within-Gibbs
- When full conditional p_m(x_m|x_{-m}) is not easy to sample from
- Replace exact Gibbs step with an MH step targeting that full conditional
- Still preserves overall stationarity

## Chapter 4 — Langevin MCMC

### Langevin SDE
- Continuous-time: dX_t = ∇log p*(X_t)dt + √2 dB_t
- Stationary distribution is p* (under mild regularity conditions)

### MALA (Metropolis-Adjusted Langevin Algorithm)
- Discretise SDE: proposal q(x'|x) = N(x + γ∇log p*(x), 2γI)
- Apply MH accept/reject → exact (targets p* exactly)

### ULA (Unadjusted Langevin Algorithm)
- Same discretisation but **no MH correction**: X_{n+1} = X_n + γ∇log p*(X_n) + √(2γ)V_n
- **Biased**: stationary distribution p^γ ≠ p*, but p^γ → p* as γ → 0
- **Gaussian analysis**: if p* = N(μ,σ²), then ULA's stationary distribution is N(μ, 2σ⁴/(2σ²−γ))
  - Requires γ < 2σ² for stability; variance inflated by factor σ²/(2σ²−γ) > 1

### §4.5.1 SGLD (Stochastic Gradient Langevin Dynamics)
- For large datasets y = (y₁,…,y_n): full gradient ∇log p* = ∇log p(x) + Σ∇log p(y_i|x) is expensive
- **SGLD**: subsample K data points, scale: gradient ≈ ∇log p(x) + (n/K)Σ_{k=1}^K ∇log p(y_{i_k}|x)
- Update: X_t = X_{t-1} + γ·(stochastic gradient) + √(2γ)V_t
- No MH correction (unbiased gradient estimate introduces additional noise)

## Chapter 4 — MCMC Diagnostics & Optimisation

### §4.6 Monitoring Convergence
- **Trace plots**: plot x_t vs t; should look like random noise around stationary mean after burn-in
- **Autocorrelation**: ρ_k = Cov(X_t, X_{t+k})/Var(X_t); want ρ_k → 0 quickly
- **ESS_MCMC** = N / (1 + 2Σ_{k=1}^∞ ρ_k); effective number of independent samples
- **Thinning**: keep every k-th sample; reduces autocorrelation but wastes computation; generally NOT recommended — better to keep all samples

### §4.7 MCMC for Optimisation

**Gradient descent as SDE discretisation**:
- Gradient descent: x_{n+1} = x_n − γ∇f(x_n) is Euler discretisation of ODE dx/dt = −∇f(x)
- Langevin for optimisation adds noise: x_{n+1} = x_n − γ∇f(x_n) + √(2γ/β)W_n

**Simulated Annealing (Algorithm 13)**:
- Target: p^β(x) ∝ exp(−βf(x)); as β→∞, p^β concentrates on global minimum of f
- **Algorithm**: MH chain on p^{β_t} with increasing β_t schedule
- Acceptance ratio: α = min(1, exp(β_t(f(x_{t-1})−f(x'))) · q(x_{t-1}|x')/q(x'|x_{t-1}))
- High β: only downhill moves accepted → refines; low β: explores broadly
- Can escape local minima (unlike gradient descent)

**Langevin for optimisation** (Eq 4.18–4.19):
- X_{n+1} = X_n − γ∇f(X_n) + √(2γ/β)W_n; β controls noise level
- Large β → low noise → concentrates near minima; small β → high noise → explores

## Chapter 5 — MMLE & EM Algorithm

### 5.1 Maximum Marginal Likelihood Estimation (MMLE)
- **Setting**: latent variable model with p_θ(x,y) = p_θ(y|x)p_θ(x); observe y, want to learn θ
- **MMLE**: θ* = argmax_θ log p_θ(y) = argmax_θ log ∫p_θ(x,y)dx
- Unlike MLE, the integral over x makes this intractable in general
- **Example 5.1**: p_θ(x)=N(θ,σ₀²), p(y|x)=N(ax,σ²) → p_θ(y)=N(aθ, a²σ₀²+σ²) → θ*=y/a

### 5.1.1 EM Algorithm
Two-step iterative method:
1. **E-step**: Q(θ, θ_{t-1}) = E_{p_{θ_{t-1}}(x|y)}[log p_θ(x,y)]
2. **M-step**: θ_t = argmax_θ Q(θ, θ_{t-1})

**Prop 5.1 (Ascent property)**: log p_{θ_t}(y) ≥ log p_{θ_{t-1}}(y)
- Proof: log p_θ(y) ≥ log p_{θ_k}(y) + Q(θ,θ_k) − Q(θ_k,θ_k) (ELBO-type bound)
- Since θ_{k+1} maximises Q, the gap Q(θ_{k+1},θ_k)−Q(θ_k,θ_k) ≥ 0

**Example 5.2**: Gaussian EM recursion θ_{k+1} = rθ_k + b where r = σ²/(σ²+a²σ₀²) ∈ (0,1)
- Affine recursion: θ_k → θ* = y/a as k→∞ (geometric convergence at rate r)

### 5.1.2 Gradient-Based MMLE
- Gradient ascent: θ_{t+1} = θ_t + δ∇_θ log p_{θ_t}(y)
- Problem: ∇log p_θ(y) involves intractable integral
- **Prop 5.2 (Fisher's identity)**: ∇_θ log p_θ(y) = E_{p_θ(x|y)}[∇_θ log p_θ(x,y)]
  - Proof: ∇log p_θ(y) = ∇p_θ(y)/p_θ(y) = ∫∇log p_θ(x,y) · p_θ(x|y)dx
- Approximate gradient with samples from p_θ(x|y): ∇log p_θ(y) ≈ (1/K)Σ∇log p_θ(x_k,y)

### 5.1.3 SOUL (Algorithm 14)
- At each θ_t: run ULA targeting p_{θ_t}(x|y) for M steps to get samples
- Use samples (after burn-in B) to estimate gradient via Fisher's identity
- Gradient ascent step: θ_{t+1} = θ_t + δ · (1/(M−B))Σ∇_θ log p_{θ_t}(X^(m),y)
- Warm start: initialise ULA chain from last sample of previous iteration

## Chapter 5 — Bayesian Parameter Estimation & PMMH

### 5.2 Bayesian Estimation
- Now put prior p(θ) on parameters: full model p(θ,x|y) ∝ p(y|x,θ)p(x|θ)p(θ)
- Goal: sample from p(θ|y) = ∫p(θ,x|y)dx (marginal posterior on parameters)

### 5.2.1 Joint Posterior Sampling (Algorithm 15)
- Run MH targeting p(θ,x|y) jointly; use θ-samples to approximate p(θ|y)
- Can also use Gibbs (alternate θ|x and x|θ) or Langevin on joint space
- Acceptance: α = min(1, p(y|x',θ')p(x'|θ')p(θ')q(θ,x|θ',x') / (p(y|x,θ)p(x|θ)p(θ)q(θ',x'|θ,x)))
- Drawback: can be expensive when x is high-dimensional

### 5.2.2 Pseudo-Marginal MH (Algorithm 16)
- **Key idea**: replace intractable p(y|θ) with unbiased IS estimate p̂(y|θ)
- Estimate: p̂(y|θ) = (1/N)Σ p(y|X_i,θ)p(X_i|θ)/q(X_i|θ), X_i ~ q(·|θ)
- PMMH acceptance: α̂ = min(1, p̂(y|θ')p(θ')q(θ|θ') / (p̂(y|θ)p(θ)q(θ'|θ)))
- Reuse p̂(y|θ) from previous iteration (only need new estimate for θ')

### Prop 5.3 — PMMH Correctness
- Define extended target: p̃*(θ,u) ∝ p(θ)m_θ(u)g(y,θ,u)
- MH on extended space with proposal q(θ'|θ)m_{θ'}(u'): acceptance = min(1, g(y,θ',u')p(θ')q(θ|θ')/(g(y,θ,u)p(θ)q(θ'|θ)))
- **Key**: m_θ(u) terms cancel in ratio!
- θ-marginal of p̃* is p(θ|y): ∫p̃*(θ,u)du ∝ p(θ)E[g(y,θ,U)] = p(θ)p(y|θ) ∝ p(θ|y)
- Hence PMMH is exact for p(θ|y), despite using noisy likelihood estimates

### Example 5.3 — Noise Analysis
- Multiplicative noise: p̂(y|θ) = p(y|θ)U with E[U]=1
- PMMH acceptance matches exact MH for very good moves (r≫1) and very bad moves (r≪1)
- Noise only affects intermediate region; higher noise c widens this region

## Chapter 6 — State-Space Models & Filtering

### 6.2 State-Space Models (SSMs)
- **Signal process** (hidden): X₀ ~ μ(x₀), X_t|X_{t-1} ~ f(x_t|x_{t-1})
- **Observation process**: Y_t|X_t ~ g(y_t|x_t)
- **Conditional independence**: Y_t depends only on X_t (not past states/observations)
- **Joint distribution**: π̄_t(x_{0:t}, y_{1:t}) = μ(x₀) ∏_{k=1}^t f(x_k|x_{k-1})g(y_k|x_k)
- **Marginal likelihood**: p(y_{1:t}) = ∫μ(x₀)∏f(x_k|x_{k-1})g(y_k|x_k) dx_{0:t}
- **Kalman filter**: exact solution when μ, f, g are all Gaussian (linear-Gaussian SSM)

### 6.2.1 Filtering Problem
- Goal: compute π_t(x_t|y_{1:t}) sequentially as new data y_t arrives
- **Predict**: ξ_t(x_t|y_{1:t-1}) = ∫f(x_t|x_{t-1})π_{t-1}(x_{t-1}|y_{1:t-1})dx_{t-1}
- **Update**: π_t(x_t|y_{1:t}) = ξ_t(x_t|y_{1:t-1}) · g(y_t|x_t) / p(y_t|y_{1:t-1})
- p(y_t|y_{1:t-1}) = ∫ξ_t(x_t|y_{1:t-1})g(y_t|x_t)dx_t (incremental marginal likelihood)

### Joint Filtering Recursion
π_t(x_{0:t}|y_{1:t}) = π_{t-1}(x_{0:t-1}|y_{1:t-1}) · f(x_t|x_{t-1})g(y_t|x_t) / p(y_t|y_{1:t-1})

This recursion is the basis for SMC methods.

## Chapter 6 — Sequential Monte Carlo & Particle Filtering

### 6.3.2 IS for SSMs — Deriving the General Particle Filter
- Target: π_t(x_{0:t}|y_{1:t}); unnormalised: π̄_t = μ(x₀)∏f(x_k|x_{k-1})g(y_k|x_k)
- Proposal: q(x_{0:t}) = q(x₀)∏q(x_k|x_{k-1}) (Markov structure)
- **Incremental weight**: W_t(x_t, x_{t-1}) = f(x_t|x_{t-1})g(y_t|x_t) / q(x_t|x_{t-1})
- **Cumulative weight** (recursive): W_{0:t} = W_{0:t-1} × W_t

### 6.3.3 SIS (Algorithm 17)
At each time t:
1. **Sample**: x̃_t^(i) ~ q(x_t|x_{t-1}^(i))
2. **Weight**: W_t^(i) = f(x̃_t^(i)|x_{t-1}^(i))g(y_t|x̃_t^(i)) / q(x̃_t^(i)|x_{t-1}^(i)); update W_{1:t}^(i) = W_{1:t-1}^(i) × W_t^(i)
3. **Normalise**: w̄_{1:t}^(i) = W_{1:t}^(i) / Σ W_{1:t}^(j)

Problem: **weight degeneracy** — after several steps, one particle has weight ≈ 1, rest ≈ 0.

### 6.3.4 SISR / General Particle Filter (Algorithm 18)
- Add **resampling** after weighting: draw x_t^(i) ~ Σ w̄_t^(j) δ_{x̃_t^(j)}
- Resampling resets weights to 1/N → only need incremental weights at next step
- Cures weight degeneracy at cost of extra variance (particle duplication)

### 6.3.5 Bootstrap Particle Filter (Algorithm 19)
- Simplest PF: set q(x_t|x_{t-1}) = f(x_t|x_{t-1}) (transition as proposal)
- **Weights simplify**: W_t^(i) = g(y_t|x̃_t^(i)) (just the likelihood!)
- Steps: (1) Propagate: x̃_t^(i) ~ f(·|x_{t-1}^(i)), (2) Weight: W_t^(i) = g(y_t|x̃_t^(i)), (3) Resample
- **Evolutionary interpretation**: propagation = mutation, weighting + resampling = selection

### 6.3.6 BPF Implementation
- Compute log-weights: log W_t^(i) = log g(y_t|x̃_t^(i))
- Normalise using logsumexp trick (Section 3.5 revisited)

### 6.3.7 Marginal Likelihood with BPF
- Decomposition: p(y_{1:t}) = ∏_{k=1}^t p(y_k|y_{1:k-1})
- BPF estimate of incremental: p̂(y_k|y_{1:k-1}) = (1/N)Σ g(y_k|x̃_k^(i))
- Full estimate: p̂(y_{1:t}) = ∏_{k=1}^t (1/N)Σ_i g(y_k|x̃_k^(i))
- **This estimator is unbiased!** (remarkable property)
- Use log-domain: log p̂(y_{1:t}) = Σ_k [log Σ_i exp(log W_k^(i)) − log N]

### 6.4 Particle MCMC
- Goal: infer θ in SSM where f_θ, g_θ depend on parameters
- Use BPF to get unbiased p̂(y_{1:T}|θ) for any θ
- Run MH: propose θ', run BPF at θ' to get p̂(y_{1:T}|θ'), accept with α = min(1, p̂(y|θ')p(θ')q(θ|θ')/(p̂(y|θ)p(θ)q(θ'|θ)))
- Correctness: same pseudo-marginal argument as PMMH (Prop 5.3) — targets exact p(θ|y_{1:T})